# Feature Selection

## 1. Import libraries and dataset

In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2, RFE
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
 
warnings.filterwarnings('ignore')

In [2]:
dataset=pd.read_csv("df.csv")

In [3]:
dataset.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices
0,Female,No,Yes,No,1,No,No,DSL,No,Yes,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,14.92,1
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,...,No,No,One year,No,Mailed check,56.95,1889.50,No,53.99,3
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,36.05,3
3,Male,No,No,No,45,No,No,DSL,Yes,No,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,40.02,3
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,50.55,1


In [4]:
# Encoding target variable
dataset['Churn'] = dataset['Churn'].map({'Yes':1, 'No':0})

In [5]:
# One-hot encode categorical columns
dataset = pd.get_dummies(dataset, drop_first=True)

In [6]:
dataset

,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,29.85,0,14.92,1,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
1,34,56.95,1889.50,0,53.99,3,True,False,False,False,...,True,False,False,False,True,False,False,False,False,True
2,2,53.85,108.15,1,36.05,3,True,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,45,42.30,1840.75,0,40.02,3,True,False,False,False,...,True,True,False,False,True,False,False,False,False,False
4,2,70.70,151.65,1,50.55,1,False,False,False,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,24,84.80,1990.50,0,79.62,7,True,False,True,True,...,True,True,True,True,True,False,True,False,False,True
7039,72,103.20,7362.90,0,100.86,6,False,False,True,True,...,True,False,True,True,True,False,True,True,False,False
7040,11,29.60,346.45,0,28.87,1,False,False,True,True,...,False,False,False,False,False,False,True,False,True,False
7041,4,74.40,306.60,1,61.32,2,True,True,True,False,...,False,False,False,False,False,False,True,False,False,True


In [7]:
# Converting all True/False columns into 0 and 1 while keeping other columns unchanged for consistency across machine learning models.

# apply(...) --> Go through each column one by one
# lambda x: --> For each column (call it x), do this
# if x.dtype == 'bool' --> Check if column is True/False type
# x.astype(int) --> Convert True → 1 and False → 0
# else x --> If not boolean, leave it as it is

dataset = dataset.apply(lambda x: x.astype(int) if x.dtype == 'bool' else x) 
dataset

,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlySpend,NumServices,gender_Male,SeniorCitizen_Yes,Partner_Yes,Dependents_Yes,...,DeviceProtection_Yes,TechSupport_Yes,StreamingTV_Yes,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1,29.85,29.85,0,14.92,1,0,0,1,0,...,0,0,0,0,0,0,1,0,1,0
1,34,56.95,1889.50,0,53.99,3,1,0,0,0,...,1,0,0,0,1,0,0,0,0,1
2,2,53.85,108.15,1,36.05,3,1,0,0,0,...,0,0,0,0,0,0,1,0,0,1
3,45,42.30,1840.75,0,40.02,3,1,0,0,0,...,1,1,0,0,1,0,0,0,0,0
4,2,70.70,151.65,1,50.55,1,0,0,0,0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,24,84.80,1990.50,0,79.62,7,1,0,1,1,...,1,1,1,1,1,0,1,0,0,1
7039,72,103.20,7362.90,0,100.86,6,0,0,1,1,...,1,0,1,1,1,0,1,1,0,0
7040,11,29.60,346.45,0,28.87,1,0,0,1,1,...,0,0,0,0,0,0,1,0,1,0
7041,4,74.40,306.60,1,61.32,2,1,1,1,0,...,0,0,0,0,0,0,1,0,0,1


## 2. Input & Output variable split

In [8]:
indep_X = dataset.drop(columns=['Churn'])    
dep_Y=dataset['Churn'] 

In [9]:
churn_table = dataset['Churn'].value_counts().to_frame(name='Count')
churn_table['Percentage (%)'] = (churn_table['Count'] / len(dataset)) * 100
churn_table['Percentage (%)'] = churn_table['Percentage (%)'].round(2)
churn_table

,Count,Percentage (%)
Churn,,
0,5174,73.46
1,1869,26.54


Since the dataset is imbalanced (74% No Churn, 26% Churn), F1-score would be a more appropriate metric for feature selection. 

## 3. Defining functions

In [10]:
def get_models():
    return {
        'Logistic'  : LogisticRegression(random_state=0, max_iter=1000),
        'SVM_Linear': SVC(kernel='linear', random_state=0),
        'SVM_RBF'   : SVC(kernel='rbf',    random_state=0),
        'KNN'       : KNeighborsClassifier(n_neighbors=5),
        'NaiveBayes': GaussianNB(),
        'Decision'  : DecisionTreeClassifier(criterion='entropy', random_state=0),
        'RandomForest': RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0),
    }
 
# Split + Scale
def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
    
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test  = sc.transform(X_test)
    return X_train, X_test, y_train, y_test
 
# Train all 7 models and return accuracy dictionary 
def train_all_models(X_train, X_test, y_train, y_test):
    results = {}
    for name, model in get_models().items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        results[name] = round(f1_score(y_test, y_pred), 4)  # swap accuracy for f1
    return results

## 4. SelectKbest method

In [11]:
print("SELECTKBEST - F1 score for each k  (Chi-Square scoring)")
print("=" * 65)

k_values   = [5, 7, 10, 12]    # List of different k values (number of features to select)      
skb_results = {}               # Dictionary to store results for each k
 
for k in k_values:      # Loop through each k value
    selector   = SelectKBest(score_func=chi2, k=k)     # Select top k important features using Chi-Square method
    X_selected = selector.fit_transform(indep_X, dep_Y)    # Fit on data and transform to keep only selected features
 
     # Get the names of selected features
    selected_cols = indep_X.columns[selector.get_support()]

     # Split data into train and test, then apply scaling
    X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)

     # Train all models and store their accuracy results
    skb_results[f'k={k}'] = train_all_models(X_train, X_test, y_train, y_test)

     # Store the selected feature names for this k
    skb_results[f'k={k}']['Selected_Features'] = list(selected_cols)   
 
# Creating a comparison table (only accuracy values, excluding feature list)
# Convert results dictionary into a clean table (rows = k values, columns = models, values = accuracy)
skb_table = pd.DataFrame(
    {k: {m: v for m, v in scores.items() if m != 'Selected_Features'}  # m-> model name, v-> accuracy value
     for k, scores in skb_results.items()}).T           # .T-> Transpose (swap rows & columns) 

# Print the comparison table
print(skb_table.to_string())
print()
 
# Print selected features for each k value
print("── Features selected by SelectKBest ──")
for k, scores in skb_results.items():
    print(f"  {k}: {scores['Selected_Features']}")
print()

SELECTKBEST - F1 score for each k  (Chi-Square scoring)
      Logistic  SVM_Linear  SVM_RBF     KNN  NaiveBayes  Decision  RandomForest
k=5     0.5197      0.5264   0.5070  0.5128      0.5583    0.4612        0.4792
k=7     0.5483      0.5290   0.5443  0.5207      0.5739    0.4728        0.4740
k=10    0.5687      0.5422   0.5851  0.5288      0.6025    0.4891        0.5000
k=12    0.5620      0.5452   0.5391  0.5231      0.6044    0.4952        0.5000

── Features selected by SelectKBest ──
  k=5: ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=7: ['tenure', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=10: ['tenure', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'TechSupport_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Electronic check']
  k=12: ['tenu

<h5 style="color:darkblue;"> 
• From the SelectKBest results, different numbers of features (k = 5, 7, 10, 12) have been tested to see which gives the best model performance.
<br> <br>

• The F1-score improves as the number of features increases up to k = 10, indicating better balance between precision and recall, and then slightly stabilizes.<br> 

• Naive Bayes achieves the highest F1-score, showing it is more effective in identifying churn customers in this imbalanced dataset.<br> 

• SVM (RBF) and Logistic Regression also perform well, providing stable and balanced performance across different feature sets.<br> 

• Key features such as tenure, MonthlyCharges, TotalCharges, Contract, and PaymentMethod consistently appear, indicating they are the most important factors influencing churn.
</h5>

## 5. RFE Method

In [12]:
# SVC(linear) and RandomForest are not used here - they are very slow inside RFE
# because RFE retrains the model once per feature eliminated.
# LogReg and DT are much faster and still give good feature rankings.

rfe_estimators = {                                                        # Define models to use inside RFE (faster models for feature ranking)
    'LogReg': LogisticRegression(max_iter=1000, random_state=0),
    'DT'    : DecisionTreeClassifier(random_state=0)}    
 
# Dictionary to store results for each k and each estimator
rfe_results = {}
 
for k in k_values:
    rfe_results[k] = {}
    for est_name, estimator in rfe_estimators.items():
        rfe        = RFE(estimator=estimator, n_features_to_select=k, step=2)  # step=2 eliminates 2 features at a time → 2x faster
        X_selected = rfe.fit_transform(indep_X, dep_Y)
 
        selected_cols = indep_X.columns[rfe.get_support()]
 
        X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)
        accs = train_all_models(X_train, X_test, y_train, y_test)
        accs['Selected_Features'] = list(selected_cols)
        rfe_results[k][est_name] = accs    # Save results for this k and estimator
 
# Formatting output for better readability
SEP  = "=" * 75
SEP2 = "-" * 75

# Loop through each k to print results
for k in k_values:
    print(f"\n{SEP}")
    print(f"  RFE  |  k = {k} features selected")
    print(SEP)
 
    # Create table of accuracies (exclude feature list)
    tbl = pd.DataFrame(
        {est: {m: f"{v:.4f}" for m, v in scores.items() if m != 'Selected_Features'}
         for est, scores in rfe_results[k].items()}).T
    
    tbl.index.name = "RFE Estimator"
    print(tbl.to_string())
 
 # Print selected features clearly
    print(f"\n  {'─'*35}")
    print(f"  Features Selected (k={k})")
    print(f"  {'─'*35}")
    
    for est, scores in rfe_results[k].items():
        feats = scores['Selected_Features']
        print(f"\n  [{est}]")
        for i, f in enumerate(feats, 1):
            print(f"    {i:>2}. {f}")
 
print(f"\n{SEP2}\n")


  RFE  |  k = 5 features selected
              Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest
RFE Estimator                                                                     
LogReg          0.5533     0.5592  0.5533  0.0723     0.5937   0.5533       0.5533
DT              0.5040     0.4858  0.4837  0.5018     0.5217   0.4286       0.4308

  ───────────────────────────────────
  Features Selected (k=5)
  ───────────────────────────────────

  [LogReg]
     1. InternetService_Fiber optic
     2. InternetService_No
     3. OnlineSecurity_Yes
     4. Contract_One year
     5. Contract_Two year

  [DT]
     1. tenure
     2. MonthlyCharges
     3. TotalCharges
     4. AvgMonthlySpend
     5. InternetService_Fiber optic

  RFE  |  k = 7 features selected
              Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest
RFE Estimator                                                                     
LogReg          0.4622     0.5592  0.4897  0.476

<h5 style="color:darkblue;"> 
• The F1-score values are generally lower compared to SelectKBest, indicating that RFE does not perform as well for this dataset, especially in identifying churn customers.<br><br> 

• Naive Bayes consistently gives the highest F1-score across all k values, showing it remains strong in detecting churn even with different feature subsets. <br>

• The performance slightly improves as k increases, with k = 10 or 12 giving better results, but the improvement is not very significant. <br>

• Two different feature patterns are observed:
  1. Logistic-based RFE selects more categorical/service-related features,
  2. Decision Tree-based RFE selects more numerical features like tenure and charges, showing different models prioritize different types of features.
</h5>

## 6. Feature Importance method

In [13]:
# Random Forest feature_importances_ ranks features by contribution.

print("FEATURE IMPORTANCE (Random Forest) - F1 score for each k")
print("=" * 65)
 
# Train one RF to get stable importance scores
rf_for_importance = RandomForestClassifier(n_estimators=100, random_state=0)
rf_for_importance.fit(indep_X, dep_Y)

# Create a DataFrame with feature names and their importance values, sorted descending
importance_df = pd.DataFrame({
    'Feature'   : indep_X.columns,
    'Importance': rf_for_importance.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Display top 10 most important features
print("\nTop 10 features by importance:")
print(importance_df.head(10).to_string(index=False))
print()

# Dictionary to store results for different k values
fi_results = {}

# Loop through different k values
for k in k_values:
    top_features   = importance_df['Feature'].head(k).tolist()   # Select top k important features
    X_selected     = indep_X[top_features]
 
    X_train, X_test, y_train, y_test = split_and_scale(X_selected, dep_Y)    # Split data and apply scaling
    fi_results[f'k={k}'] = train_all_models(X_train, X_test, y_train, y_test)   # Train all models and store accuracy
    fi_results[f'k={k}']['Selected_Features'] = top_features     # Store selected feature names

# Convert results into comparison table (exclude feature list)
fi_table = pd.DataFrame(
    {k: {m: v for m, v in scores.items() if m != 'Selected_Features'}
     for k, scores in fi_results.items()}).T

# Print accuracy table
print(fi_table.to_string())
print()

# Print selected features for each k

print("── Features selected by Feature Importance ──")

for k, scores in fi_results.items():
    print(f"  {k}: {scores['Selected_Features']}")

print()

FEATURE IMPORTANCE (Random Forest) - F1 score for each k

Top 10 features by importance:
                       Feature  Importance
                  TotalCharges    0.163939
                MonthlyCharges    0.143985
                        tenure    0.137736
               AvgMonthlySpend    0.137544
   InternetService_Fiber optic    0.039991
PaymentMethod_Electronic check    0.038668
                   NumServices    0.034098
             Contract_Two year    0.033764
                   gender_Male    0.024558
          PaperlessBilling_Yes    0.023157

      Logistic  SVM_Linear  SVM_RBF     KNN  NaiveBayes  Decision  RandomForest
k=5     0.5040      0.4858   0.4837  0.5018      0.5217    0.4291        0.4464
k=7     0.5052      0.4859   0.5072  0.5294      0.5574    0.4984        0.4805
k=10    0.5096      0.4885   0.5098  0.5247      0.5680    0.4666        0.4955
k=12    0.5351      0.5105   0.5340  0.5352      0.5803    0.4928        0.5156

── Features selected by Feature Impo

<h5 style="color:darkblue;"> 
• The F1-score gradually improves as the number of features increases, with k = 12 giving the best performance, indicating that including more relevant features improves the model’s ability to detect churn. <br>  <br> 

• Naive Bayes consistently achieves the highest F1-score, showing it is more effective in handling imbalanced data and identifying churn customers. <br> 

• Compared to SelectKBest, the overall performance is slightly lower, indicating that Feature Importance provides good insights but may not be the most optimal feature selection method for this dataset. <br> 

• The most important features are TotalCharges, MonthlyCharges, tenure, and AvgMonthlySpend, showing that customer spending and duration are the key drivers of churn.
</h5>

## 7. Feature Selection - Summary table

In [14]:
# SUMMARY TABLE  (best accuracy across all methods)
# best method + k combination = the row with the highest accuracies across models  
    
print("=" * 65)
print("SUMMARY - Best F1 score per method (max over k values)")
print("=" * 65)
 
model_cols = list(get_models().keys())  # Get model names (columns)
 
summary_rows = {}   # Dictionary to store best results for each method
 
# SelectKBest best row
summary_rows['SKB_best'] = skb_table[model_cols].max().to_dict()    # Take maximum accuracy for each model across all k values
summary_rows['SKB_best']['best_k'] = skb_table[model_cols].mean(axis=1).idxmax()  # Find k value with highest average performance
 
# Flatten RFE results into one table (combine all k + estimators)
rfe_flat_rows = {}

for k in k_values:
    for est in rfe_estimators:
        label = f"RFE_k{k}_{est}"  # label like RFE_k10_LogReg
        rfe_flat_rows[label] = {m: rfe_results[k][est][m] for m in model_cols}

# Convert to DataFrame
rfe_flat_df = pd.DataFrame(rfe_flat_rows).T

# Take best accuracy across all combinations
summary_rows['RFE_best'] = rfe_flat_df[model_cols].max().to_dict()

# Find best combination (k + estimator)
summary_rows['RFE_best']['best_k'] = rfe_flat_df[model_cols].mean(axis=1).idxmax()
 
# Feature Importance best row
summary_rows['FI_best'] = fi_table[model_cols].max().to_dict()  # Take best accuracy across k values
summary_rows['FI_best']['best_k'] = fi_table[model_cols].mean(axis=1).idxmax()  # Find best k

# Convert everything into final summary table
grand_summary = pd.DataFrame(summary_rows).T

print(grand_summary.to_string())  # Print final comparison
print()

SUMMARY - Best F1 score per method (max over k values)
         Logistic SVM_Linear SVM_RBF     KNN NaiveBayes Decision RandomForest          best_k
SKB_best   0.5687     0.5452  0.5851  0.5288     0.6044   0.4952          0.5            k=10
RFE_best    0.562     0.5592  0.5533  0.5234     0.5963   0.5533       0.5533  RFE_k10_LogReg
FI_best    0.5351     0.5105   0.534  0.5352     0.5803   0.4984       0.5156            k=12



<h5 style="color:darkblue;"> 
• SelectKBest performs the best overall, with the highest F1-scores across most models, especially at k = 10, making it the most effective feature selection method for this dataset.<br><br>

• Naive Bayes achieves the highest F1-score across all methods, confirming it is the best model for identifying churn in this imbalanced dataset.<br>

• RFE gives comparable but slightly lower performance, with its best results coming from Logistic-based feature selection at k = 10, showing moderate effectiveness.<br>

• Feature Importance performs the weakest among the three methods, even though performance improves at k = 12, indicating it is less optimal for this problem compared to SelectKBest.
</h5>

# Model Creation

### 1. Selecting Final Best Features

In [15]:
# Select best 10 features (based on summary result)
selector = SelectKBest(score_func=chi2, k=10)  

# Apply feature selection on full dataset
X_best = selector.fit_transform(indep_X, dep_Y)

# Get selected feature names
selected_features = indep_X.columns[selector.get_support()]
print("Selected Features:", list(selected_features))

Selected Features: ['tenure', 'MonthlyCharges', 'TotalCharges', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_Yes', 'TechSupport_Yes', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Electronic check']


### 2. Train-Test Split

In [16]:
# Split dataset into train and test (final split for model building)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_best, dep_Y, test_size=0.25, stratify=dep_Y, random_state=0)

# stratify=dep_Y --> Stratified splitting ensures that both training and testing datasets maintain the same target variable distribution. 
# It is important for reliable evaluation in imbalanced datasets.

### 3. Apply SMOTE (only on training)

Dataset is imbalanced → models are biased toward predicting No Churn.<br>
Solution: Apply SMOTE to balance the training data before model training (oversampling method).<br>
SMOTE creates synthetic Churn training samples so the model trains on a 50/50 split. Usually gives the biggest recall improvement. <br>
One important rule: SMOTE only touches training data, never the test set.

In [17]:
# Import SMOTE
# !pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance churn classes in training data
smote = SMOTE(random_state=0)
X_train, y_train = smote.fit_resample(X_train, y_train)

### 4. Scaling

In [18]:
# Apply scaling (important for distance-based models like SVM, KNN)
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()

X_train = sc.fit_transform(X_train)   # fit on training data
X_test  = sc.transform(X_test)        # apply same scaling to test data

### 5. Defining Models

In [19]:
# Defining Models with their hyperparameter grids

models_params = {'Logistic Regression': {'model' : LogisticRegression(random_state=0, max_iter=1000),
                                         'params': {'C': [0.01, 0.1, 1, 10], 'penalty': ['l2'], 'solver' : ['lbfgs', 'liblinear']}},
 
'SVM (RBF)': {'model' : SVC(random_state=0),'params': {'C': [0.1, 1, 10],'gamma': ['scale', 'auto'],'kernel': ['rbf']}},
 
'SVM (Linear)': {'model' : SVC(random_state=0),'params': {'C'     : [0.01, 0.1, 1, 10],'kernel': ['linear']}},
 
'KNN': {'model' : KNeighborsClassifier(),'params': {'n_neighbors': [3, 5, 7, 9], 'metric': ['minkowski', 'euclidean']}},
 
'Naive Bayes': {'model' : GaussianNB(),'params': {'var_smoothing': [1e-9, 1e-8, 1e-7]}},
 
'Decision Tree': {'model' : DecisionTreeClassifier(random_state=0),'params': {'criterion': ['entropy', 'gini'],'max_depth': [None, 5, 10],
                                                                              'min_samples_split': [2, 5]}},
 
'Random Forest': {'model' : RandomForestClassifier(random_state=0),'params': {'n_estimators': [50, 100],'criterion': ['entropy', 'gini'],
                                                                              'max_depth'   : [None, 5, 10]}}}

### 6. GridSearch (FINAL MODEL TRAINING)

In [20]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

results = []

# Loop through all models
for name, mp in models_params.items():
    
    # Create GridSearch object
    gs = GridSearchCV(
        estimator=mp['model'],
        param_grid=mp['params'],
        cv=5,
        scoring='f1',   # f1 score
        n_jobs=-1)
    
    # Train model
    gs.fit(X_train, y_train)
    
    # Predict on test data
    y_pred = gs.best_estimator_.predict(X_test)
    
    # Calculate F1 score (since dataset is imbalanced)
    test_f1 = f1_score(y_test, y_pred)   
    
    # Store results
    results.append({
        'Model': name,
        'Best Params': gs.best_params_,
        'CV F1 Score': round(gs.best_score_, 4),
        'Test F1 Score': round(test_f1, 4)})
    
    # Print model results
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Best params   : {gs.best_params_}")
    print(f"  CV F1 score   : {gs.best_score_:.4f}")
    print(f"  Test F1 score : {test_f1:.4f}")
    
    print(f"\n{classification_report(y_test, y_pred, target_names=['No Churn', 'Churn'])}")


  Logistic Regression
  Best params   : {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}
  CV F1 score   : 0.7729
  Test F1 score : 0.6174

              precision    recall  f1-score   support

    No Churn       0.91      0.72      0.80      1294
       Churn       0.50      0.80      0.62       467

    accuracy                           0.74      1761
   macro avg       0.71      0.76      0.71      1761
weighted avg       0.80      0.74      0.75      1761


  SVM (RBF)
  Best params   : {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
  CV F1 score   : 0.8154
  Test F1 score : 0.6106

              precision    recall  f1-score   support

    No Churn       0.87      0.82      0.85      1294
       Churn       0.57      0.65      0.61       467

    accuracy                           0.78      1761
   macro avg       0.72      0.74      0.73      1761
weighted avg       0.79      0.78      0.78      1761


  SVM (Linear)
  Best params   : {'C': 0.1, 'kernel': 'linear'}
  CV F1 score   : 

### 7. Final Comparison Table

In [21]:
final_results = pd.DataFrame(results)

# Reorder columns for clarity
final_results = final_results[['Model', 'CV F1 Score', 'Test F1 Score', 'Best Params']]

# Round values
final_results[['CV F1 Score', 'Test F1 Score']] = final_results[['CV F1 Score', 'Test F1 Score']].round(4)

print(final_results)

                 Model  CV F1 Score  Test F1 Score  \
0  Logistic Regression       0.7729         0.6174   
1            SVM (RBF)       0.8154         0.6106   
2         SVM (Linear)       0.7625         0.5919   
3                  KNN       0.8014         0.5412   
4          Naive Bayes       0.7750         0.6090   
5        Decision Tree       0.8118         0.5550   
6        Random Forest       0.8311         0.6083   

                                         Best Params  
0       {'C': 1, 'penalty': 'l2', 'solver': 'lbfgs'}  
1        {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}  
2                     {'C': 0.1, 'kernel': 'linear'}  
3          {'metric': 'minkowski', 'n_neighbors': 3}  
4                           {'var_smoothing': 1e-09}  
5  {'criterion': 'gini', 'max_depth': 10, 'min_sa...  
6  {'criterion': 'entropy', 'max_depth': 10, 'n_e...  


<h5 style="color:green;"> 
- CV F1 Score (Cross Validation) tells - How well will model perform in general? <br>
- Test F1 Score tells - How well does it perform on new data?</h5>

<h5 style="color:darkblue;"> 
• Best Model Selection: Logistic Regression is the best performing model because it achieved the highest Test F1 Score (0.6174). While other models had higher training scores, Logistic Regression generalizes best to unseen data.<br> <br>
</h5>

### 8. Save the best model & objects

In [22]:
import pickle

# Save model
filename = "churn_model.sav"
pickle.dump(gs.best_estimator_, open(filename, 'wb'))  # save best model

In [23]:
# Save scaler
pickle.dump(sc, open("scaler.sav", 'wb'))

In [24]:
# Save feature selection
pickle.dump(selector, open("selector.sav", 'wb'))

In [25]:
# Save column names
pickle.dump(indep_X.columns, open("model_columns.sav", 'wb'))